# Ученик на CE: серия запусков (Colab, T4)

Шаг 3 проекта `cifar100-distillation`: выбранная по сетке ёмкости конфигурация `SimpleCNN`
обучается `N_RUNS` раз из разных начальных состояний одной задачей ClearML
(`train.py --n_runs`, класс `Experiment` в `kd/experiment.py`).

- Начальные веса запуска `i` задаёт сид, порождённый из `--init_seed`;
- порядок батчей, аугментации и прочая случайность — общий `--seed`, одинаковый для всех
  запусков.

Каждый запуск пишет в `<DRIVE_OUT>/<task_id>/run_<i>/`. После обрыва сессии тот же
набор аргументов с `--continue_task_id` пропустит завершённые запуски и продолжит
прерванный с чекпоинта.

Перед запуском: **Среда выполнения → T4 GPU**, в **Secrets** — `CLEARML_API_ACCESS_KEY`
и `CLEARML_API_SECRET_KEY`.

## 1. Google Drive и GPU

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Нет GPU: смени среду выполнения на T4"

## 2. Код и зависимости

torch/torchvision в Colab уже стоят со сборкой под CUDA — их не переустанавливаем,
ставим только остальное из `requirements.txt`. Обучение идёт отдельным процессом
(`!python train.py`), поэтому предупреждения pip о версиях пакетов самого Colab не мешают.

In [ ]:
import os

REPO_URL = "https://github.com/esgon191/cifar100-distillation"
REPO_DIR = "/content/cifar100-distillation"
BRANCH = "master"

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
!git log -1 --oneline

In [ ]:
!grep -viE '^(torch|torchvision)==' requirements.txt > /tmp/requirements-colab.txt
!pip install -q -r /tmp/requirements-colab.txt

## 3. ClearML

Ключи берутся из Colab Secrets и передаются через переменные окружения — их наследует
процесс `train.py` (`Task.set_credentials` подействовал бы только на ядро ноутбука).

In [ ]:
from google.colab import userdata

for name in ("CLEARML_API_ACCESS_KEY", "CLEARML_API_SECRET_KEY"):
    os.environ[name] = userdata.get(name)
os.environ["CLEARML_API_HOST"] = "https://api.clear.ml"
os.environ["CLEARML_WEB_HOST"] = "https://app.clear.ml"
os.environ["CLEARML_FILES_HOST"] = "https://files.clear.ml"

# Проверка авторизации без создания задачи
!python -c "from clearml.backend_api.session import Session; Session(); print('ClearML OK, files server:', Session.get_files_server_host())"

## 4. Параметры серии

`CNN_WIDTH`/`CNN_CONVS` — конфигурация, выбранная по сетке ёмкости (см. `docs/student.md`).
Рецепт тот же, что в сетке: 100 эпох, SGD nesterov lr 0.1, wd 5e-4, батч 128, warmup 1, косинус.

In [ ]:
DRIVE_OUT = "/content/drive/MyDrive/cifar100-distillation/out"
CNN_WIDTH, CNN_CONVS = 32, 2
N_RUNS, INIT_SEED, SEED = 10, 0, 0

TRAIN_ARGS = (
    f"--arch cnn --cnn_width {CNN_WIDTH} --cnn_convs_per_stage {CNN_CONVS} "
    f"--n_runs {N_RUNS} --init_seed {INIT_SEED} --seed {SEED} "
    f"--task_name student-ce-w{CNN_WIDTH}-c{CNN_CONVS}-x{N_RUNS} --tags student-ce baseline "
    "--num_train_epochs 100 --learning_rate 0.1 --weight_decay 5e-4 --warmup_epochs 1 "
    "--per_device_train_batch_size 128 --save_every 5 "
    f"--dataset_name cifar100 --output_dir {DRIVE_OUT}"
)
os.makedirs(DRIVE_OUT, exist_ok=True)
print(TRAIN_ARGS)

## 5. Запуск серии

Запиши id задачи из строки `ClearML Task: created new task id=...`.

In [ ]:
!python -u train.py {TRAIN_ARGS}

## 6. Продолжение после обрыва сессии

Ячейки 1–4, затем эта с id задачи. Завершённые запуски (есть `run_<i>/summary.json`)
пропускаются, прерванный продолжается с `run_<i>/checkpoint.pt`. Чтобы расширить серию
(например, до 20 запусков), достаточно поднять `N_RUNS` с тем же `INIT_SEED`: первые
запуски не изменятся.

In [ ]:
TASK_ID = ""  # id задачи ClearML = имя папки в DRIVE_OUT
assert TASK_ID, "Укажи TASK_ID"
!python -u train.py {TRAIN_ARGS} --continue_task_id {TASK_ID}

## 7. После обучения

Single values `test/<метрика>` — среднее по запускам, `test/<метрика>_std` — стандартное
отклонение; таблица по запускам — артефакт `runs`.

In [ ]:
drive.flush_and_unmount()